<a href="https://colab.research.google.com/github/diwakarasd/Test/blob/main/AIHelperHub_Keyword_NLP_mapping_with_URLs_in_sitemap_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install trafilatura

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import trafilatura
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
url_counter = 1;
def parse_sitemap(sitemap_url):
    """
    Parses an XML sitemap file and returns a list of non-image URLs.
    """

    try:
        response = requests.get(sitemap_url)
        response.raise_for_status()  # Raise an exception for non-200 status codes
        print(f"inside function ")
        # Check for successful response
        if response.status_code == 200:
            global url_counter;
            soup = BeautifulSoup(response.content, 'xml')
            urls = []
            image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.svg')
            exclude_keywords = ["news", "careers","partners","integrations","product-updates","company","business-templates","video-team"]
            for url_tag in soup.find_all('loc'):
                url = url_tag.text.strip()
                if not url.lower().endswith(image_extensions)and not any(keyword in url for keyword in exclude_keywords):  # Check for image file extensions
                    #if "/search/" in url and "?" not in url:
                      urls.append(url)

            for url in urls:
                print(f"URL {url_counter} found: {url}")
                url_counter +=1

            return urls
        else:
            print(f"Error retrieving sitemap: Status code {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error fetching sitemap from {sitemap_url}: {e}")
        return None

# Replace 'https://www.example.com/sitemap.xml' with the actual sitemap URL
sitemap_url1 = "https://www.hubspot.com/sitemap.xml"


# Call the parse_sitemap function to retrieve URLs from the sitemap
urls = parse_sitemap(sitemap_url1)



inside function 
URL 1 found: https://www.hubspot.com/comparisons/sales-thankyou
URL 2 found: https://www.hubspot.com/blog/bid/31882/hubspot-launches-new-conversion-assists-report
URL 3 found: https://www.hubspot.com/case-studies/triaster
URL 4 found: https://www.hubspot.com/blog/bid/5284/hubspot-authors-create-book-grader-to-track-success-of-inbound-marketing-book
URL 5 found: https://www.hubspot.com/pt/roi-calculator-embed-test
URL 6 found: https://www.hubspot.com/blog/bid/34205/hubspot-continues-rapid-trajectory-with-82-revenue-growth
URL 7 found: https://www.hubspot.com/resources/courses/customer-service
URL 8 found: https://www.hubspot.com/insights
URL 9 found: https://www.hubspot.com/resources/kit/visual-design
URL 10 found: https://www.hubspot.com/resources/courses/other
URL 11 found: https://www.hubspot.com/blog/bid/18569/hubspot-customers-also-winning-awards
URL 12 found: https://www.hubspot.com/resources/webinar/branding
URL 13 found: https://www.hubspot.com/products/service/

In [4]:
## Create a requests session with a larger connection pool
session = requests.Session()
adapter = HTTPAdapter(pool_connections=100, pool_maxsize=100)
session.mount('http://', adapter)
session.mount('https://', adapter)
counter = 1;
def fetch_text_from_url(session, url):
    print(f"Processing URL: {url}")  # Print the URL being processed
    global counter;
    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            text = trafilatura.extract(downloaded)
            if text:
                print(f"URL No {counter} Successfully fetched text from {url}")  # Print on successful fetch
                counter += 1
                return text
            else:
                print(f"Failed to extract text from {url}.")
                return None
        else:
            print(f"Failed to download content from {url}.")
            return None
    except Exception as e:
        print(f"Error fetching text from URL: {e}")
        return None

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = ''.join([char for char in text if char.isalpha() or char.isspace()])
    # Tokenize text
    words = word_tokenize(text)
    # Remove stopwords
    stop_words_english = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words_english]
    # Rejoin words into a single string
    text = ' '.join(words)
    return text

def extract_texts(urls):
    documents = []

    # Use ThreadPoolExecutor to fetch texts in parallel
    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_url = {executor.submit(fetch_text_from_url, session, url): url for url in urls}
        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                text = future.result()
                if text:
                    print(f"Processing text for {url}")  # Print before processing text
                    processed_text = preprocess_text(text)
                    documents.append((url, processed_text))
                else:
                    print(f"Failed to retrieve text from {url}. Skipping.")
            except Exception as e:
                print(f"Error processing {url}: {e}")

    if not documents:
        print("No text retrieved from any URL. Exiting.")
        return None

    return documents


if urls:
    documents = extract_texts(urls)
else:
    print("Failed to retrieve URLs from the sitemap.")

Processing URL: https://www.hubspot.com/comparisons/sales-thankyouProcessing URL: https://www.hubspot.com/company-news/hubspots-inbound-2013-conference-kicks-off-today-in-downtown-boston
Processing URL: https://www.hubspot.com/blog/bid/31882/hubspot-launches-new-conversion-assists-report

Processing URL: https://www.hubspot.com/case-studies/triaster
Processing URL: https://www.hubspot.com/company-news/hubspot-launches-new-community-for-black-business-professionals
Processing URL: https://www.hubspot.com/partner-news/april-2018-tier-promotions
Processing URL: https://www.hubspot.com/blog/bid/5284/hubspot-authors-create-book-grader-to-track-success-of-inbound-marketing-book
Processing URL: https://www.hubspot.com/pt/roi-calculator-embed-test
Processing URL: https://www.hubspot.com/product-updates/audience-sync
Processing URL: https://www.hubspot.com/blog/bid/34205/hubspot-continues-rapid-trajectory-with-82-revenue-growth


URL No 1 Successfully fetched text from https://www.hubspot.com/partner-news/april-2018-tier-promotions
Processing URL: https://www.hubspot.com/resources/courses/customer-service
Processing text for https://www.hubspot.com/partner-news/april-2018-tier-promotions
URL No 2 Successfully fetched text from https://www.hubspot.com/company-news/hubspot-launches-new-community-for-black-business-professionals
Processing URL: https://www.hubspot.com/apac/newsroom/hubspot-launches-first-australian-data-centre
Processing text for https://www.hubspot.com/company-news/hubspot-launches-new-community-for-black-business-professionals


URL No 3 Successfully fetched text from https://www.hubspot.com/company-news/hubspots-inbound-2013-conference-kicks-off-today-in-downtown-boston
Processing URL: https://www.hubspot.com/insights
Processing text for https://www.hubspot.com/company-news/hubspots-inbound-2013-conference-kicks-off-today-in-downtown-boston
URL No 4 Successfully fetched text from https://www.hubspot.com/blog/bid/31882/hubspot-launches-new-conversion-assists-report
Processing URL: https://www.hubspot.com/resources/kit/visual-design
Processing text for https://www.hubspot.com/blog/bid/31882/hubspot-launches-new-conversion-assists-report
URL No 5 Successfully fetched text from https://www.hubspot.com/blog/bid/5284/hubspot-authors-create-book-grader-to-track-success-of-inbound-marketing-book
Processing URL: https://www.hubspot.com/product-updates/february-2019-integration-roundup
Processing text for https://www.hubspot.com/blog/bid/5284/hubspot-authors-create-book-grader-to-track-success-of-inbound-marketing-book

URL No 6 Successfully fetched text from https://www.hubspot.com/comparisons/sales-thankyou
Processing URL: https://www.hubspot.com/resources/courses/other
Processing text for https://www.hubspot.com/comparisons/sales-thankyou
URL No 7 Successfully fetched text from https://www.hubspot.com/case-studies/triaster
Processing URL: https://www.hubspot.com/blog/bid/18569/hubspot-customers-also-winning-awards
Processing text for https://www.hubspot.com/case-studies/triaster


URL No 8 Successfully fetched text from https://www.hubspot.com/pt/roi-calculator-embed-test
Processing URL: https://www.hubspot.com/partner-news/hubspot-crm-now-with-ads-and-email
Processing text for https://www.hubspot.com/pt/roi-calculator-embed-test
URL No 9 Successfully fetched text from https://www.hubspot.com/product-updates/audience-sync
Processing URL: https://www.hubspot.com/resources/webinar/branding
Processing text for https://www.hubspot.com/product-updates/audience-sync
URL No 10 Successfully fetched text from https://www.hubspot.com/insights
Processing URL: https://www.hubspot.com/products/service/issue-tracking
Processing text for https://www.hubspot.com/insights
URL No 11 Successfully fetched text from https://www.hubspot.com/resources/courses/customer-service
Processing URL: https://www.hubspot.com/startups/stories/black-founders/diversd
Processing text for https://www.hubspot.com/resources/courses/customer-service


...

Failed to download content from https://www.hubspot.com/careers-blog/building-an-operating-system-for-work/life-balance.
Processing URL: https://www.hubspot.com/case-studies/ceros-sales-hub
Failed to retrieve text from https://www.hubspot.com/careers-blog/building-an-operating-system-for-work/life-balance. Skipping.


ERROR:trafilatura.downloads:download error: https://www.hubspot.com/company-news/hubspot-ecosystem-white-paper HTTPSConnectionPool(host='www.hubspot.com', port=443): Max retries exceeded with url: /company-news/hubspot-ecosystem-white-paper (Caused by ResponseError('too many 429 error responses'))
ERROR:trafilatura.downloads:download error: https://www.hubspot.com/case-studies/consafe-logistics HTTPSConnectionPool(host='www.hubspot.com', port=443): Max retries exceeded with url: /case-studies/consafe-logistics (Caused by ResponseError('too many 429 error responses'))


Failed to download content from https://www.hubspot.com/company-news/hubspot-ecosystem-white-paper.
Processing URL: https://www.hubspot.com/blog/bid/22161/italian-students-learn-about-inbound-marketing
Failed to retrieve text from https://www.hubspot.com/company-news/hubspot-ecosystem-white-paper. Skipping.
Failed to download content from https://www.hubspot.com/case-studies/consafe-logistics.
Processing URL: https://www.hubspot.com/case-studies/whitby-school


Failed to retrieve text from https://www.hubspot.com/case-studies/consafe-logistics. Skipping.


KeyboardInterrupt: 

In [5]:
def vectorize_documents(documents):
    """
    Vectorizes the documents using CountVectorizer.
    Returns URLs and vectorized representations of texts.
    """
    urls, texts = zip(*documents)  # Unzip the tuples into separate lists
    # Vectorize the documents
    print("Vectorizing documents...")  # Print when starting vectorization
    vectorizer = CountVectorizer(stop_words='english')  # Adjust parameters as needed
    text_vectorized = vectorizer.fit_transform(texts)
    return urls, text_vectorized, vectorizer

def vectorize_keyword(keyword, vectorizer):
    """
    Vectorizes the given keyword using the same vectorizer as the documents.
    """
    return vectorizer.transform([keyword])

def group_urls_by_keyword(urls, text_vectorized, vectorizer, keyword, similarity_threshold):
    """
    Groups URLs based on cosine similarity with a given keyword.
    Prints URLs that have a similarity score above the threshold.
    """
    print("Vectorizing keyword...")  # Print when vectorizing keyword
    keyword_vectorized = vectorize_keyword(keyword, vectorizer)

    print("Calculating similarity scores...")  # Print when calculating similarity
    similarities = cosine_similarity(text_vectorized, keyword_vectorized).flatten()

    # Filter URLs that have similarity above the threshold
    matched_urls = [urls[i] for i in range(len(urls)) if similarities[i] >= similarity_threshold]

    # Print the grouped URLs
    if matched_urls:
        print("\nURLs matching the given keyword based on cosine similarity:")
        for url in matched_urls:
            print(f" - {url}")
    else:
        print("\nNo URLs matched the given keyword above the threshold.")


if documents:
    urls, text_vectorized, vectorizer = vectorize_documents(documents)
    similarity_threshold = 0.1  # Adjust the similarity threshold here (e.g., 70%)
    group_urls_by_keyword(urls, text_vectorized, vectorizer,"inbound marketing", similarity_threshold)
else:
    print("Failed to retrieve URLs from the sitemap.")

NameError: name 'documents' is not defined